# 18 — Debugging nonlinear solvers systematically

**Objectives.** Separate formulation, initialization, evaluation, and convergence failures; read return statuses; inspect infeasibilities and KKT residuals.

A useful order is: (1) confirm shapes/units and finite values, (2) evaluate every residual at the initial guess, (3) inspect variable and constraint bounds, (4) scale variables/residuals/objective, (5) verify derivatives, (6) inspect sparsity and rank, (7) read solver statistics, then (8) change solver options.


In [1]:
# Make repository-local helpers importable without installing a package.
import sys
from pathlib import Path

repository_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "casadi_tutorial").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

# Autoreload picks up edits to tutorial models before each notebook cell runs.
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

### Diagnose an intentionally infeasible optimization problem

The cell creates a scalar Opti problem with contradictory bounds. IPOPT is
expected to fail, so the solve is wrapped in `try/except`. The helper
`solver_summary` extracts stable status fields, while
`opti.debug.show_infeasibilities` evaluates the constraints at the last
iterate and reports violations above the requested tolerance.

The problem is

$$
\begin{equation}
\min_x (x-2)^2
\quad\text{subject to}\quad
x\ge3,\qquad x\le1.
\end{equation}
$$

Its feasible set is empty because no real number can satisfy both
inequalities. The example demonstrates that solver failure is information
to inspect rather than an exception to silently suppress.


In [ ]:
import casadi as ca

from casadi_tutorial.diagnostics import solver_summary

optimization = ca.Opti()
x = optimization.variable()
optimization.minimize((x - 2) ** 2)

# These constraints are intentionally inconsistent for the debugging demo.
optimization.subject_to(x >= 3)
optimization.subject_to(x <= 1)
optimization.set_initial(x, 2)
optimization.solver(
    "ipopt",
    {"print_time": False},
    {"print_level": 0, "max_iter": 20},
)

try:
    optimization.solve()
except RuntimeError as error:
    print("expected failure:", str(error).splitlines()[-1])
    print(solver_summary(optimization.stats()))
    optimization.debug.show_infeasibilities(1e-8)

### Detect invalid numerical values during function evaluation

`ca.Function` accepts the option `regularity_check=True`, which raises an
error if an evaluation produces `NaN` or infinity. The cell deliberately
evaluates a square root outside its real domain and confirms that the
domain error is surfaced.

The function is

$$
\begin{equation}
f(z)=\sqrt{z-1},
\end{equation}
$$

which is real-valued only when

$$
\begin{equation}
z-1\ge0\quad\Longleftrightarrow\quad z\ge1.
\end{equation}
$$

At $z=0$, the expression is invalid over the reals. Regularity checks
are particularly useful for locating problematic iterates in logarithms,
divisions, fractional powers, and physical property models.


In [3]:
argument = ca.MX.sym("argument")

# regularity_check makes Function evaluation raise when NaN or Inf appears.
dangerous_function = ca.Function(
    "dangerous_function",
    [argument],
    [ca.sqrt(argument - 1)],
    {"regularity_check": True},
)
try:
    dangerous_function(0)
except RuntimeError as error:
    print("expected domain failure captured:", type(error).__name__)

expected domain failure captured: RuntimeError


## Status-driven actions

- **Infeasible/restoration failed:** inspect bounds and residual blocks; consider explicit, penalized slack only when violation has physical meaning.
- **Maximum iterations:** check progress, scaling, derivatives, and initialization before increasing the limit.
- **Invalid number:** isolate the reported output row, evaluate dependencies, and protect model domains physically.
- **Search direction/singular system:** inspect redundant constraints, Jacobian rank, curvature, and nonsmooth expressions.
- **Acceptable rather than desired tolerance:** calculate application-level residuals before accepting it.

`Opti.debug.x_describe(i)` and `g_describe(i)` map vector indices back to source. IPOPT's derivative checker, limited-memory Hessian, bound relaxation, and verbosity are diagnostic tools—not permanent patches for a broken model. **Recap:** preserve the smallest failing case and report versions, status, statistics, bounds, initial values, and residuals.
